# Run the complete mytriton test suite in Google Colab

## 1. Select the revision

Change `REVISION` to a branch, tag, or commit. When this notebook is opened inside a local checkout, that checkout is used directly and `REVISION` is ignored.

In [ ]:
REPOSITORY_URL = "https://github.com/pbelevich/mytriton.git"
REVISION = "main"
REQUIRE_SM80 = True

## 2. Find or clone the repository

The clone goes into a fresh temporary directory, so rerunning this cell does not overwrite another checkout.

In [17]:
import subprocess
import sys
import tempfile
from pathlib import Path


def run(command, *, env=None):
    print("+", " ".join(command), flush=True)
    with subprocess.Popen(
        command,
        cwd=repo,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()

    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def find_mytriton_checkout():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "mytriton"
        ).is_dir():
            return candidate
    return None


repo = find_mytriton_checkout()
if repo is None:
    clone_root = Path(tempfile.mkdtemp(prefix="mytriton-colab-"))
    repo = clone_root / "mytriton"
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REVISION,
            REPOSITORY_URL,
            str(repo),
        ],
        check=True,
    )

print("Repository:", repo)
subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=repo,
    check=True,
)
subprocess.run(
    ["git", "log", "-1", "--oneline", "--decorate"],
    cwd=repo,
    check=True,
)

Repository: /tmp/mytriton-colab-nlmkf99r/mytriton


CompletedProcess(args=['git', 'log', '-1', '--oneline', '--decorate'], returncode=0)

## 3. Install test and CUDA dependencies

PyTorch is already included in Colab GPU runtimes. Its CUDA build selects the matching `cuda12` or `cuda13` project extra, avoiding multiple incompatible CuPy packages in one environment.

In [18]:
import torch

if torch.version.cuda is None:
    raise RuntimeError("The installed PyTorch build does not include CUDA")
cuda_major = int(torch.version.cuda.split(".", maxsplit=1)[0])
if cuda_major not in (12, 13):
    raise RuntimeError(f"Unsupported CUDA major version: {cuda_major}")
cuda_extra = f"cuda{cuda_major}"
print(f"PyTorch uses CUDA {torch.version.cuda}; installing {cuda_extra}")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        f"{repo}[{cuda_extra},dev]",
    ],
    check=True,
)

source_root = str(repo / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Installed mytriton and its CUDA/development dependencies")

PyTorch uses CUDA 12.8; installing cuda12
Installed mytriton and its CUDA/development dependencies


## 4. Validate CUDA and BF16 support

`REQUIRE_SM80=True` makes the notebook fail early on a GPU that cannot execute native BF16 MMA. Set it to `False` to run the rest of the suite on an older CUDA GPU.

In [19]:
import cupy as cp
import ml_dtypes

device = cp.cuda.Device()
properties = cp.cuda.runtime.getDeviceProperties(device.id)
device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()
compute_capability = int(device.compute_capability)

print("Python:", sys.version.split()[0])
print("GPU:", device_name)
print(
    "Compute capability:",
    f"{compute_capability // 10}.{compute_capability % 10}",
)
print("CuPy:", cp.__version__)
print("PyTorch:", torch.__version__)
print("ml_dtypes:", ml_dtypes.__version__)
print("PyTorch BF16 support:", torch.cuda.is_bf16_supported())

assert torch.cuda.is_available(), "PyTorch cannot see a CUDA GPU"
assert compute_capability >= 75, "FP16 m16n8k8 MMA requires sm_75+"
if REQUIRE_SM80:
    assert compute_capability >= 80, "Native BF16 MMA requires sm_80+"
    assert torch.cuda.is_bf16_supported()

Python: 3.13.15
GPU: NVIDIA A100-SXM4-40GB
Compute capability: 8.0
CuPy: 14.0.1
PyTorch: 2.11.0+cu128
ml_dtypes: 0.6.0
PyTorch BF16 support: True


## 5. Run the test suite

If full MLIR bindings are installed, one `pytest` invocation runs every test. Otherwise one invocation runs the suite with `-k "not mlir"`, excluding tests whose node IDs contain `mlir`.

In [20]:
import os

from mytriton.mlir_backend import mlir_available

test_env = os.environ.copy()
test_env["MYTRITON_REQUIRE_CUDA"] = "1"
test_env["PYTHONUNBUFFERED"] = "1"
pytest_command = [sys.executable, "-m", "pytest", "-v", "-s", "-ra"]

if mlir_available():
    print("MLIR bindings found: running every test")
    run(pytest_command, env=test_env)
else:
    print(
        "MLIR bindings are unavailable; running all unit/codegen tests "
        "and all CUDA execution tests"
    )
    run(
        [*pytest_command, "-k", "not mlir"],
        env=test_env,
    )

MLIR bindings are unavailable; running all unit/codegen tests and all CUDA execution tests
+ /usr/bin/python3 -m pytest -v -s -ra -k not mlir
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /tmp/mytriton-colab-nlmkf99r/mytriton
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.12.1, typeguard-4.6.0, anyio-4.14.2
collecting ... collected 521 items / 10 deselected / 511 selected

tests/test_add_kernel.py::test_add_kernel_codegen[cuda] PASSED
tests/test_add_kernel.py::test_add_kernel_execution[cuda] PASSED
tests/test_ast_runtime_loop.py::test_runtime_for_preserves_source_order_for_multiple_carried_values PASSED
tests/test_ast_runtime_loop.py::test_runtime_for_executes_multiple_carried_values PASSED
tests/test_ast_runtime_loop.py::test_runtime_for_supports_annotated_carried_assignment PASSED
tests/test_ast_runtime_loop.py::test_ru